In [3]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

def evaluate_model(model, X, y, cv=5):
    """
    Evaluates a given model using cross-validation with specified metrics.

    Parameters:
    model: A machine learning model (e.g., RandomForestClassifier, SVC, etc.)
    X: Input feature data (numpy array or pandas DataFrame)
    y: Target labels (numpy array or pandas Series)
    cv: Number of cross-validation folds (default is 5)

    Returns:
    dict: Dictionary containing the mean values of accuracy, ROC-AUC, precision, recall, and F1-score across the cross-validation folds.
    """

    # Define the scoring metrics
    scoring = {
        'accuracy': make_scorer(accuracy_score),
        'roc_auc': make_scorer(roc_auc_score),
        'precision': make_scorer(precision_score),
        'recall': make_scorer(recall_score),
        'f1': make_scorer(f1_score)
    }

    # Perform cross-validation
    cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring)

    # Compute the mean of each metric across all folds
    results = {
        'accuracy': cv_results['test_accuracy'].mean(),
        'roc_auc': cv_results['test_roc_auc'].mean(),
        'precision': cv_results['test_precision'].mean(),
        'recall': cv_results['test_recall'].mean(),
        'f1': cv_results['test_f1'].mean()
    }

    return results




In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

coffee_data = pd.read_csv(r"..\data\reduced_data_num.csv")

# Create the target column
coffee_data['target'] = coffee_data['caffeine_class'].map(lambda x: 0 if x == 'Absent' else 1)

# Store the column names before scaling
column_names = coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']).columns

scaler = StandardScaler()
X_scaled_std = scaler.fit_transform(coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']))
X = X_scaled_std.copy()  # Your scaled data
y = coffee_data['target']

In [5]:
# Assuming X and y are your input data and target, and model is your machine learning model

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Call the function to evaluate the model
results = evaluate_model(rf_model, X_train, y_train, cv=5)

# Print the results
print("Cross-validation results (mean values):")
for metric, value in results.items():
    print(f"{metric.capitalize()}: {value:.4f}")

Cross-validation results (mean values):
Accuracy: 0.9402
Roc_auc: 0.9068
Precision: 0.9471
Recall: 0.8324
F1: 0.8846


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# Assuming X and y are your input data and target
# X_train, X_test, y_train, y_test already defined in your previous context

# Initialize and fit the Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model to the training data
rf_model.fit(X_train, y_train)

# Get predicted probabilities
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]

# Adjust the threshold for classification (e.g., 0.4 instead of 0.5)
threshold = 0.4
y_pred_adjusted = (y_pred_prob >= threshold).astype(int)

# Recalculate metrics with the adjusted threshold
print(f"Precision: {precision_score(y_test, y_pred_adjusted):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_adjusted):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_adjusted):.4f}")


Precision: 0.9211
Recall: 0.7609
F1-Score: 0.8333


In [8]:
# Adjust the threshold for classification again (e.g., 0.35)
threshold = 0.35
y_pred_adjusted = (y_pred_prob >= threshold).astype(int)

# Recalculate metrics with the new adjusted threshold
print(f"Precision: {precision_score(y_test, y_pred_adjusted):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_adjusted):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_adjusted):.4f}")


Precision: 0.9211
Recall: 0.7609
F1-Score: 0.8333


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

models = [
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('Support Vector Classifier', SVC(probability=True, random_state=42)),
    ('K-Nearest Neighbors', KNeighborsClassifier()),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Naive Bayes', GaussianNB())
]

# Loop through each model and apply the evaluation function
for name, model in models:
    print(f"Evaluating {name}...")
    results = evaluate_model(model, X_train, y_train, cv=5)
    
    # Print the results for each model
    print(f"Results for {name}:")
    for metric, value in results.items():
        print(f"{metric.capitalize()}: {value:.4f}")
    print("\n" + "-"*40 + "\n")

Evaluating Random Forest...
Results for Random Forest:
Accuracy: 0.9402
Roc_auc: 0.9068
Precision: 0.9471
Recall: 0.8324
F1: 0.8846

----------------------------------------

Evaluating Logistic Regression...
Results for Logistic Regression:
Accuracy: 0.8450
Roc_auc: 0.7548
Precision: 0.8300
Recall: 0.5548
F1: 0.6625

----------------------------------------

Evaluating Support Vector Classifier...
Results for Support Vector Classifier:
Accuracy: 0.8885
Roc_auc: 0.8433
Precision: 0.8410
Recall: 0.7429
F1: 0.7860

----------------------------------------

Evaluating K-Nearest Neighbors...
Results for K-Nearest Neighbors:
Accuracy: 0.8614
Roc_auc: 0.8303
Precision: 0.7518
Recall: 0.7619
F1: 0.7536

----------------------------------------

Evaluating Decision Tree...
Results for Decision Tree:
Accuracy: 0.9401
Roc_auc: 0.9094
Precision: 0.9440
Recall: 0.8414
F1: 0.8869

----------------------------------------

Evaluating Naive Bayes...
Results for Naive Bayes:
Accuracy: 0.6140
Roc_auc: 

In [17]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

coffee_data = pd.read_csv(r"..\data\reduced_data_num.csv")

# Create the target column
coffee_data['target'] = coffee_data['caffeine_class'].map(lambda x: 0 if x == 'Absent' else 1)

# Store the column names before scaling
column_names = coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']).columns

scaler = StandardScaler()
X_scaled_std = scaler.fit_transform(coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']))
X = X_scaled_std.copy()  # Your scaled data
y = coffee_data['caffeine_percent']

In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
import numpy as np

from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import cross_validate
import numpy as np

def adjusted_r2(r2, n, p):
    """
    Calculate the Adjusted R² score.
    
    Parameters:
    r2: R² score
    n: Number of samples
    p: Number of features
    
    Returns:
    Adjusted R² score
    """
    return 1 - ((1 - r2) * (n - 1)) / (n - p - 1)

# Define a generalized evaluation function for regression models
def evaluate_regression_model(model, X, y, cv=5):
    """
    Evaluates a regression model using cross-validation with specified metrics.
    
    Parameters:
    model: A regression model (e.g., RandomForestRegressor, SVR, etc.)
    X: Input feature data (numpy array or pandas DataFrame)
    y: Target values (numpy array or pandas Series)
    cv: Number of cross-validation folds (default is 5)

    Returns:
    dict: Dictionary containing the mean values of MAE, MSE, RMSE, and R-squared across the cross-validation folds.
    """

        # Define the scoring metrics for regression
    scoring = {
        'mae': make_scorer(mean_absolute_error, greater_is_better=False),
        'mse': make_scorer(mean_squared_error, greater_is_better=False),
        'r2': make_scorer(r2_score),
        'explained_variance': make_scorer(explained_variance_score)
    }

    # Perform cross-validation
    cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, return_estimator=True)

    # Compute RMSE from MSE
    rmse_values = np.sqrt(-cv_results['test_mse'])

    # Compute Adjusted R² for each fold using the individual R² values, number of samples, and features
    n_samples = X.shape[0]
    n_features = X.shape[1]
    adjusted_r2_values = [adjusted_r2(r2, n_samples, n_features) for r2 in cv_results['test_r2']]

    # Compute the mean of each metric across all folds
    results = {
        'MAE': -cv_results['test_mae'].mean(),
        'MSE': -cv_results['test_mse'].mean(),
        'RMSE': rmse_values.mean(),
        'R2': cv_results['test_r2'].mean(),
        'Adjusted R2': np.mean(adjusted_r2_values),
        'Explained Variance': cv_results['test_explained_variance'].mean()
    }

    return results




In [23]:
# Define a list of regression models to evaluate
models = [
    ('Random Forest Regressor', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('Linear Regression', LinearRegression()),
    ('Support Vector Regressor', SVR()),
    ('K-Nearest Neighbors Regressor', KNeighborsRegressor()),
    ('Decision Tree Regressor', DecisionTreeRegressor(random_state=42))
]

# Assuming X and y are your input data and target for regression
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Loop through each model and apply the evaluation function
for name, model in models:
    print(f"Evaluating {name}...")
    results = evaluate_regression_model(model, X_train, y_train, cv=5)
    
    # Print the results for each model
    print(f"Results for {name}:")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")
    print("\n" + "-"*40 + "\n")

Evaluating Random Forest Regressor...
Results for Random Forest Regressor:
MAE: 0.0096
MSE: 0.0028
RMSE: 0.0351
R2: -0.2086
Adjusted R2: -0.2673
Explained Variance: -0.1250

----------------------------------------

Evaluating Linear Regression...
Results for Linear Regression:
MAE: 0.0163
MSE: 0.0029
RMSE: 0.0385
R2: -0.7215
Adjusted R2: -0.8051
Explained Variance: -0.6091

----------------------------------------

Evaluating Support Vector Regressor...
Results for Support Vector Regressor:
MAE: 0.0387
MSE: 0.0045
RMSE: 0.0624
R2: -8.9306
Adjusted R2: -9.4130
Explained Variance: -5.3256

----------------------------------------

Evaluating K-Nearest Neighbors Regressor...
Results for K-Nearest Neighbors Regressor:
MAE: 0.0098
MSE: 0.0027
RMSE: 0.0313
R2: 0.3444
Adjusted R2: 0.3126
Explained Variance: 0.3539

----------------------------------------

Evaluating Decision Tree Regressor...
Results for Decision Tree Regressor:
MAE: 0.0083
MSE: 0.0028
RMSE: 0.0335
R2: 0.1102
Adjusted R2: 0

In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split

# Assuming X and y are your input data and target for regression
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define the regression models to evaluate
models = [
    ('Random Forest Regressor', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('Linear Regression', LinearRegression()),
    ('Support Vector Regressor', SVR()),
    ('K-Nearest Neighbors Regressor', KNeighborsRegressor()),
    ('Decision Tree Regressor', DecisionTreeRegressor(random_state=42))
]

# Loop through each model and apply the evaluation function
for name, model in models:
    print(f"Evaluating {name}...")
    results = evaluate_regression_model(model, X_train, y_train, cv=5)
    
    # Print the results for each model
    print(f"Results for {name}:")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")
    print("\n" + "-"*40 + "\n")

Evaluating Random Forest Regressor...
Results for Random Forest Regressor:
MAE: 0.0096
MSE: 0.0028
RMSE: 0.0351
R2: -0.2086
Adjusted R2: -0.2673
Explained Variance: -0.1250

----------------------------------------

Evaluating Linear Regression...
Results for Linear Regression:
MAE: 0.0163
MSE: 0.0029
RMSE: 0.0385
R2: -0.7215
Adjusted R2: -0.8051
Explained Variance: -0.6091

----------------------------------------

Evaluating Support Vector Regressor...
Results for Support Vector Regressor:
MAE: 0.0387
MSE: 0.0045
RMSE: 0.0624
R2: -8.9306
Adjusted R2: -9.4130
Explained Variance: -5.3256

----------------------------------------

Evaluating K-Nearest Neighbors Regressor...
Results for K-Nearest Neighbors Regressor:
MAE: 0.0098
MSE: 0.0027
RMSE: 0.0313
R2: 0.3444
Adjusted R2: 0.3126
Explained Variance: 0.3539

----------------------------------------

Evaluating Decision Tree Regressor...
Results for Decision Tree Regressor:
MAE: 0.0083
MSE: 0.0028
RMSE: 0.0335
R2: 0.1102
Adjusted R2: 0

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Load the dataset
coffee_data = pd.read_csv(r"..\data\reduced_data_num.csv")

# Create the target column
coffee_data['target'] = coffee_data['caffeine_class'].map(lambda x: 0 if x == 'Absent' else 1)

# Store the column names before scaling
column_names = coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']).columns

# Scale the features
scaler = StandardScaler()
X_scaled_std = scaler.fit_transform(coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']))

# Define X and y
y = coffee_data['target']
X = X_scaled_std

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Get feature importances
importances = model.feature_importances_

# Use the stored column names to map the importances back to the feature names
important_features = sorted(zip(column_names, importances), key=lambda x: x[1], reverse=True)
feature_importance_dict = {feature: importance for feature, importance in zip(column_names, importances)}

# Display the most important features
for feature, importance in important_features:
    print(f"Feature: {feature}, Importance: {importance:.4f}")


Feature: clim_14_tmax2_feb, Importance: 0.1031
Feature: clim_25_prec1_jan, Importance: 0.0836
Feature: clim_34_prec10_oct, Importance: 0.0800
Feature: clim_27_prec3_mar, Importance: 0.0747
Feature: clim_1_tmin1_jan, Importance: 0.0673
Feature: clim_40_temp_season, Importance: 0.0657
Feature: clim_69_cwd_annual, Importance: 0.0654
Feature: clim_39_isotherm, Importance: 0.0591
Feature: clim_35_prec11_nov, Importance: 0.0570
Feature: clim_28_prec4_apr, Importance: 0.0564
Feature: clim_26_prec2_feb, Importance: 0.0540
Feature: clim_38_mean_diurn_range, Importance: 0.0486
Feature: clim_13_tmax1_jan, Importance: 0.0454
Feature: env_74_solrad, Importance: 0.0427
Feature: env_72_slo, Importance: 0.0366
Feature: env_79_forcov, Importance: 0.0309
Feature: env_73_asp, Importance: 0.0294
